# Case Disruption — the CD index and the F/E/G decomposition

Funk & Owen-Smith's CD index for each case, plus the Funk–E–G decomposition, in fixed windows.
Reads `cache/case_csr.npz` from **case_metadata**. Primary key: `case_id`.

## The metric

For a focal case *F*, look at everything that cites *F* (**A**) and everything that cites *F*'s
own references without citing *F* (**B**):

| symbol | who |
|---|---|
| `ni` | cites *F* but **none** of *F*'s references — *F* stands alone for that citer |
| `nj` | cites *F* **and** at least one of its references — *F* is consolidating |
| `nk` | cites *F*'s references but **not** *F* — the prior work stands on its own |

$$\mathrm{CD} = \frac{n_i - n_j}{n_i + n_j + n_k}$$

so CD → +1 is disruptive and → −1 is consolidating.

`F`, `E`, `G` — **Foundation, Extension, Generalization** — partition the same citers a second
way. For a citer *c*, let $e_j$ be how many of *F*'s **references** *c* also cites and $e_i$ how
many of *F*'s **citers** *c* also cites:

| share | condition | reading |
|---|---|---|
| `E` extension | $e_j > e_i$ | *c* leans on the ground *F* itself stood on — *F* is being extended along the same line |
| `F` foundation | $e_i > e_j$ | *c* leans on what came **after** *F* rather than before it |
| `G` generalization | $e_j = e_i = 0$ | *c* takes *F* on its own, touching neither side |

Ties ($e_j = e_i > 0$) are split half and half between `E` and `F`, so `F + E + G = 1` for
every case with a citer. Naming follows `patent_disruption` and the `cext` / `cfound`
counters in the engine below, so the three families report the same quantity under the same
name.

Windows `3, 5, 10, all` bound `year(citer) - year(F)`; a same-year citation is inside every
window. A case with no citers has CD undefined and stays `NaN` — not 0, which would be a
number that means "perfectly balanced".

## Output
`Case law/output/case_disruption.parquet` — `case_id` plus `CD/F/E/G/ni/nj/nk` × `{3,5,10,all}`

## Cost, and why this one is a job and not a login-node cell

The engine is a Python loop over all 5,179,698 cases, and its per-case work is dominated by
assembling **B** — the union of the citers of *F*'s references. Measured over this graph, the
sum of in-degrees across a case's references is **6,004 on average and 478,842 at worst**, for
**31.1 billion** element touches in total, roughly nine times the equivalent load in
`patent_disruption`. The worst single case is only ~4 MB of int32, so memory is never the
problem; time is. Submit it.

In [1]:
import os, sys, gc, time
import numpy as np, pandas as pd
from tqdm.auto import tqdm
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Case law')
import cl_common as cl
OUT_FP = cl.out('case_disruption.parquet')
WINDOWS = [3, 5, 10, -1]
WIN_SFX = {3: '_3', 5: '_5', 10: '_10', -1: '_all'}
cl.preflight('case_disruption')
out_ptr, out_idx, in_ptr, in_idx, year, uni = cl.load_csr()
n = len(uni)
print(f'cases {n:,}   edges {len(out_idx):,}')

case law : /project/jevans/Dawoon/Science of Science/Case law
output   : /project/jevans/Dawoon/Science of Science/Case law/output
cache    : /project/jevans/Dawoon/Science of Science/Case law/cache

  case_disruption             OK
CSR cache present: /project/jevans/Dawoon/Science of Science/Case law/cache/case_csr.npz
cases 5,179,698   edges 47,519,638


## Engine

Identical in structure to `PatentView/notebook/patent_disruption.ipynb` — the same CSR walk,
the same `isin_sorted` membership test, the same reduceat segment sums — so the two metric
families are the same estimator on different graphs rather than two implementations that happen
to share a name.

In [2]:
def isin_sorted(x, s):
    """Membership of x in the SORTED array s, without building a hash set."""
    if len(s) == 0 or len(x) == 0:
        return np.zeros(len(x), bool)
    i = np.searchsorted(s, x)
    i = np.clip(i, 0, len(s) - 1)
    return s[i] == x


def compute(out_ptr, out_idx, in_ptr, in_idx, year, windows, n):
    keys = ['cd', 'f', 'e', 'g', 'ni', 'nj', 'nk']
    res = {w: {k: np.full(n, np.nan, np.float32) for k in keys} for w in windows}
    for F in tqdm(range(n), desc='CD+FEG', mininterval=10):
        a0, a1 = in_ptr[F], in_ptr[F + 1]
        if a1 == a0:
            continue                                   # no citers -> undefined
        A = in_idx[a0:a1]; yF = year[F]; yA = year[A]
        refs = out_idx[out_ptr[F]:out_ptr[F + 1]]; Rset = np.sort(refs)
        # refs of each citer, concatenated with a segment offset per citer
        lens = out_ptr[A + 1] - out_ptr[A]; tot = int(lens.sum())
        seg = np.zeros(len(A), np.int64); np.cumsum(lens[:-1], out=seg[1:])
        pos = np.arange(tot) - np.repeat(seg, lens) + np.repeat(out_ptr[A], lens)
        RC = out_idx[pos]
        ej = (np.add.reduceat(isin_sorted(RC, Rset).astype(np.int64), seg) if tot
              else np.zeros(len(A), np.int64))
        if len(refs):
            B = np.unique(np.concatenate([in_idx[in_ptr[r]:in_ptr[r + 1]] for r in refs]))
            B = B[B != F]
        else:
            B = np.empty(0, np.int64)
        yB = year[B] if len(B) else np.empty(0, np.int64)
        for w in windows:
            if w == -1:
                inwin = yA - yF >= 0
                mB = (yB - yF >= 0) if len(B) else np.zeros(0, bool)
            else:
                inwin = (yA - yF >= 0) & (yA - yF <= w)
                mB = ((yB - yF >= 0) & (yB - yF <= w)) if len(B) else np.zeros(0, bool)
            Aw = A[inwin]; N = len(Aw); Bw = int(mB.sum())
            if N == 0 and Bw == 0:
                continue
            Cset = np.sort(Aw)
            ei = (np.add.reduceat(isin_sorted(RC, Cset).astype(np.int64), seg) if tot
                  else np.zeros(len(A), np.int64))
            up = ej[inwin]; down = ei[inwin]
            cext = int((up > down).sum()); cfound = int((down > up).sum())
            tie = int(((up == down) & (up > 0)).sum()); cg = int(((up == 0) & (down == 0)).sum())
            nj = int((up > 0).sum()); ni = N - nj; nk = Bw - nj
            denom = ni + nj + nk
            r = res[w]
            r['ni'][F] = ni; r['nj'][F] = nj; r['nk'][F] = nk
            r['cd'][F] = (ni - nj) / denom if denom > 0 else np.nan
            if N > 0:
                r['e'][F] = (cext + 0.5 * tie) / N
                r['f'][F] = (cfound + 0.5 * tie) / N
                r['g'][F] = cg / N
    return res


print('engine ready — CD + F/E/G, windows', WINDOWS)

engine ready — CD + F/E/G, windows [3, 5, 10, -1]


In [3]:
%%time
t0 = time.time()
res = compute(out_ptr, out_idx, in_ptr, in_idx, year, WINDOWS, n)
print(f'computed in {(time.time()-t0)/3600:.2f} h')

cols = {'case_id': uni}
for w in WINDOWS:
    s = WIN_SFX[w]
    cols[f'CD{s}'] = res[w]['cd']
    cols[f'F{s}']  = res[w]['f'];  cols[f'E{s}'] = res[w]['e'];  cols[f'G{s}'] = res[w]['g']
    cols[f'ni{s}'] = res[w]['ni']; cols[f'nj{s}'] = res[w]['nj']; cols[f'nk{s}'] = res[w]['nk']
dis = pd.DataFrame(cols)
dis.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(dis):,} rows, {len(dis.columns)} cols, '
      f'{os.path.getsize(OUT_FP)/1e6:.0f} MB)')
for w in WINDOWS:
    s = WIN_SFX[w]
    print(f'  CD{s:<5} defined {dis[f"CD{s}"].notna().mean()*100:5.1f}%  '
          f'mean {dis[f"CD{s}"].mean():+.4f}  |  '
          f'f/e/g = {dis[f"F{s}"].mean():.3f}/{dis[f"E{s}"].mean():.3f}/{dis[f"G{s}"].mean():.3f}')
display(dis.head(5))

computed in 0.28 h
WROTE /project/jevans/Dawoon/Science of Science/Case law/output/case_disruption.parquet  (5,179,698 rows, 29 cols, 170 MB)
  CD_3    defined  67.2%  mean +0.0463  |  f/e/g = 0.052/0.548/0.400
  CD_5    defined  68.5%  mean +0.0579  |  f/e/g = 0.071/0.524/0.405
  CD_10   defined  70.1%  mean +0.0759  |  f/e/g = 0.102/0.488/0.411
  CD_all  defined  73.1%  mean +0.1210  |  f/e/g = 0.172/0.412/0.416


,case_id,CD_3,F_3,E_3,G_3,ni_3,nj_3,nk_3,CD_5,F_5,E_5,G_5,ni_5,nj_5,nk_5,CD_10,F_10,E_10,G_10,ni_10,nj_10,nk_10,CD_all,F_all,E_all,G_all,ni_all,nj_all,nk_all
0,1,0.0,NaN,NaN,NaN,0.0,0.0,24.0,0.0,NaN,NaN,NaN,0.0,0.0,39.0,0.00,NaN,NaN,NaN,0.0,0.0,71.0,-0.017241,0.000000,1.000000,0.0,0.0,3.0,171.0
1,2,0.0,NaN,NaN,NaN,0.0,0.0,2.0,-0.5,0.0,1.0,0.0,0.0,2.0,2.0,-0.25,0.0,1.0,0.0,0.0,2.0,6.0,-0.080000,0.000000,1.000000,0.0,0.0,2.0,23.0
2,3,0.0,NaN,NaN,NaN,0.0,0.0,4.0,0.0,NaN,NaN,NaN,0.0,0.0,10.0,0.00,NaN,NaN,NaN,0.0,0.0,25.0,-0.049180,0.333333,0.666667,0.0,0.0,3.0,58.0
3,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
